# Módulo 1. Selección e integración de información
**Responsable:** Abdelmouiz kobry  
**Asignatura:** Minería de Datos (UCLM)  

Objetivo: verificar la procedencia de una copia pública del CSV, seleccionar variables y crear una vista contextual con indicadores INE, SIIU y ESTUDES. No se generan registros sustitutivos ni se mezclan estadísticas nacionales con variables individuales para entrenamiento.

**Preparación previa:** instalar las dependencias con `python -m pip install -r requirements.txt` y descargar el CSV con `python fuente_auditoria.py --descargar` desde el directorio del módulo. La identidad exacta del espejo con la versión de Kaggle no se ha certificado.


In [ ]:
from pathlib import Path
import pandas as pd
from fuente_auditoria import ARCHIVO_LOCAL, cargar_y_auditar
from seleccion_normalizacion import seleccionar_y_normalizar, variables_predictoras
from integracion_contextual import integrar_contexto, COLUMNAS_CONTEXTO
from ejecutar_modulo1 import ejecutar

if not ARCHIVO_LOCAL.is_file():
    raise FileNotFoundError('Descarga primero el CSV público con: python fuente_auditoria.py --descargar')

datos, auditoria = cargar_y_auditar(ARCHIVO_LOCAL, exigir_version_referencia=True)
print('Filas:', auditoria['filas'], '| Columnas:', auditoria['columnas'])
print('Nulos:', {k:v for k,v in auditoria['nulos'].items() if v})
print('Distribución de Overall_Impact:', auditoria['clases'])


## Selección y normalización
`Student_ID` se elimina como predictor. Se armonizan unas etiquetas categóricas, preservando objetivos y valores faltantes. La imputación y el escalado quedan para los módulos posteriores, aplicados al entrenamiento.


In [ ]:
base = seleccionar_y_normalizar(datos)
assert len(base) == len(datos) and 'Student_ID' not in base.columns
print('Columnas de base:', len(base.columns))
print('Predictores:', variables_predictoras(base))
display(base.head(5))


## Vista exploratoria de contexto
Los indicadores nacionales de España se incorporan mediante uniones `LEFT JOIN` para contextualizar los grupos comparables. No constituyen medidas personales y NO deben formar parte del conjunto de predictores.


In [ ]:
contexto, evid = integrar_contexto(base)
assert len(contexto) == len(base)
print('Cobertura de indicadores:', evid['cobertura'])
display(contexto[['Age','Academic_Level',*COLUMNAS_CONTEXTO]].head(8))


## Exportación y trazabilidad
Se generan dos CSV separados: base para módulos 2 a 5 y referencia ecológica solo exploratoria, más el JSON de auditoría. Los archivos de salida están excluidos del control de versiones.


In [ ]:
evidencia = ejecutar(ARCHIVO_LOCAL, Path('outputs'), version_publica=True)
print('Salida creada en outputs/')
print('Autor:', evidencia['autor'])


## Advertencias para el equipo
- Confirmar la procedencia original y el método de recogida del conjunto Kaggle.
- Revisar la definición de `Overall_Impact` antes de predecirla con GPA y variables psicométricas.
- La fuerte desigualdad de las tres clases exige medidas como Macro F1 y evaluación de la clase minoritaria.
- Al cambiar la fuente, recalcular el análisis exploratorio, las transformaciones, PCA y los modelos anteriores.
- El informe completo del grupo debe seguir el formato ACM (2 a 8 páginas) y presentarse el 27/11/2026.
